# 🐦‍⬛ RAVEN-2 — Google Colab Pre-Training Master Notebook
### Ultra-Scale 10GB Cloud-Generated Personality + 40GB Factual Intelligence (50+ GB Hybrid Corpus)

Welcome to the official **Raven-2** pre-training notebook. This notebook is fully optimized for **Google Colab (A100 or T4 GPUs)** and handles streaming data download, massive 10GB personality corpus generation directly in the cloud, vocabulary tokenization, binary compilation, and auto-resume training with Google Drive backups.

---

## 🛠️ Step 0: Set Up Environment & Mount Google Drive
This mounts your Google Drive so that all checkpoints, trained tokenizers, and datasets are backed up in real time. If your Colab session times out or restarts, you can resume training in seconds.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Create target directories on your Google Drive
import os
os.makedirs("/content/drive/MyDrive/Raven2/checkpoints", exist_ok=True)
os.makedirs("/content/drive/MyDrive/Raven2/data", exist_ok=True)

# 3. Symlink Google Drive directories to our workspace for easy, high-speed access
!rm -rf data checkpoints
!ln -s "/content/drive/MyDrive/Raven2/data" "./data"
!ln -s "/content/drive/MyDrive/Raven2/checkpoints" "./checkpoints"

print("✅ Google Drive successfully mounted and project directories linked!")

### Install Required High-Performance Dependencies
Install the optimized Rust-backed tokenizers and raw dataset handlers.

In [ ]:
# Install dependencies
!pip install datasets tqdm tokenizers numpy -q

# Verify GPU status
import torch
print("=========================================================")
print("CUDA GPU Active:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
print("=========================================================")

---
## 📚 Step 1: Download the 40GB High-Intelligence Dataset
This streams and processes FineWeb-Edu, Wikipedia, and Cosmopedia v2, filtering out noise and keeping high-quality text.

In [ ]:
# Run the dataset downloader to fetch the 40GB intelligent knowledge corpus
!python build_dataset.py --target_gb 40

---
## 🛠️ Step 2: Combine Local raw corpora (Optional)
If you have uploaded corpus files to your `data` folder (like `corpus1.txt` through `corpus4.txt`), this combines them.

In [ ]:
# Merges any local corpus1.txt to corpus4.txt files uploaded to data/
!python build_dataset.py --combine

---
## 🎭 Step 3: Cloud-Generate the 10GB Personality Corpus
Uploading a 10GB file from your local computer can take hours. Instead, **generate it directly inside Google Colab's cloud environment at extreme speed!**

This generates exactly **19,200,000 conversational dialogue pairs** using our premium, non-repetitive dry sarcastic templater, resulting in exactly **10.7 GB** of high-entropy personality dataset written streamingly.

In [ ]:
# Generate 19.2M conversational dialogue pairs (10GB+ personality dataset)
!python build_personality_corpus.py --count 19200000

---
## 🔀 Step 4: Unified Corpus Concatenation
Now, concatenate your cloud-generated 10.7 GB personality corpus, your 1.3 GB local combined corpus, and the 40 GB knowledge corpus into the ultimate **52 GB training corpus** (`data/corpus.txt`).

In [ ]:
# Merge all sources into the final master corpus
# 1. Rename local combined to prevent name collision
if os.path.exists("data/combined.txt"):
    !mv data/combined.txt data/local_combined.txt

# 2. Append local combined (if exists) and the huge personality dataset into the main corpus
print("Merging corpora into data/corpus.txt...")
if os.path.exists("data/local_combined.txt"):
    !cat data/local_combined.txt >> data/corpus.txt
if os.path.exists("data/personality.txt"):
    !cat data/personality.txt >> data/corpus.txt

# 3. Verify final sizes
print("\n=============================================")
print("FINAL PRE-TRAINING CORPUS LIBRARIES:")
print("=============================================")
!ls -lh data/
print("=============================================")

---
## 🔤 Step 5: Train the Custom 16K BPE Tokenizer
Train the Byte-Level BPE tokenizer directly on the final 52 GB combined dataset. It will learn optimal subwords for English, code, markdown templates, and custom conversational alignment.

In [ ]:
# Train BPE Tokenizer on the full 52GB corpus
!python tokenizer_train.py --input data/corpus.txt

---
## 📦 Step 6: Binary Pack & Compile Corpus (Memmap)
Convert the raw text dataset into binary `uint16` tokens. The script streams data from disk and processes it in chunks, making it **fully RAM-safe** even on standard T4 GPUs.

In [ ]:
# 1. Compile pre-training files (generates train.bin and val.bin)
!python data_prep.py --mode pretrain

# 2. Compile instruction-tuning files (generates instruct.bin)
!python data_prep.py --mode instruct

---
## 🚀 Step 7: Launch RAVEN-2 Pre-Training!
Initiate the pre-training loop. The script uses Flash Attention, AMP Mixed Precision, and Google Drive auto-checkpointing (fully resume-safe). 

**To keep Colab from disconnecting, run the keep-alive script in your browser console (F12):**
```javascript
function ClickConnect(){
  console.log("Keep-alive clicked");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
```

In [ ]:
# Start training!
!python pretrain.py